# Defense Rankings Ingestion from FantasyPros

**Purpose:** Fetch weekly DST (Defense/Special Teams) rankings from FantasyPros and make them available via API.

**Source:** https://www.fantasypros.com/nfl/rankings/dst-cheatsheets.php

**Schedule:** Run weekly (Tuesday mornings after MNF)

**Output Table:** `main.fantasai.defense_rankings_current`

**Features:**
* Expert consensus rankings
* Tier classifications
* Weekly opponent matchups
* Fantasy point projections
* Best/worst ranks

In [0]:
%pip install requests beautifulsoup4 lxml --quiet

## ⚠️ Scraping Challenge

FantasyPros loads rankings dynamically with JavaScript, which requires a browser automation tool like Selenium.

**Alternative Solutions:**
1. **Manual Input** - Update rankings weekly via this notebook (quick solution below)
2. **Selenium/Playwright** - Set up browser automation (requires additional setup)
3. **FantasyPros API** - Use official API if available (requires API key/subscription)
4. **Alternative Source** - ESPN, NFL.com, or other sources with static HTML

**Recommended:** Use manual input for now, then migrate to API or browser automation later.

## ⚠️ Scraping Challenge

FantasyPros loads rankings dynamically with JavaScript, which requires a browser automation tool like Selenium.

**Alternative Solutions:**
1. **Manual Input** - Update rankings weekly via this notebook (quick solution below)
2. **Selenium/Playwright** - Set up browser automation (requires additional setup)
3. **FantasyPros API** - Use official API if available (requires API key/subscription)
4. **Alternative Source** - ESPN, NFL.com, or other sources with static HTML

**Recommended:** Use manual input for now, then migrate to API or browser automation later.

In [0]:
# Manual Defense Rankings Input
# Update this list weekly with latest rankings from FantasyPros or other source

from datetime import datetime
import pandas as pd

print("="*80)
print("Defense Rankings Ingestion - Manual Update")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*80)

# Sample rankings - UPDATE THESE WEEKLY
# Source: https://www.fantasypros.com/nfl/rankings/dst-cheatsheets.php
defense_rankings_manual = [
    {'rank': 1, 'team_abbr': 'BAL', 'team_name': 'Baltimore Ravens', 'opponent': 'vs DEN', 'best_rank': 1, 'worst_rank': 3, 'avg_rank': 1.8},
    {'rank': 2, 'team_abbr': 'SF', 'team_name': 'San Francisco 49ers', 'opponent': 'vs GB', 'best_rank': 1, 'worst_rank': 5, 'avg_rank': 2.5},
    {'rank': 3, 'team_abbr': 'DAL', 'team_name': 'Dallas Cowboys', 'opponent': 'vs NYG', 'best_rank': 2, 'worst_rank': 6, 'avg_rank': 3.2},
    {'rank': 4, 'team_abbr': 'BUF', 'team_name': 'Buffalo Bills', 'opponent': 'vs MIA', 'best_rank': 3, 'worst_rank': 7, 'avg_rank': 4.1},
    {'rank': 5, 'team_abbr': 'KC', 'team_name': 'Kansas City Chiefs', 'opponent': 'vs LAC', 'best_rank': 3, 'worst_rank': 8, 'avg_rank': 5.3},
    {'rank': 6, 'team_abbr': 'CLE', 'team_name': 'Cleveland Browns', 'opponent': 'vs PIT', 'best_rank': 4, 'worst_rank': 10, 'avg_rank': 6.4},
    {'rank': 7, 'team_abbr': 'PHI', 'team_name': 'Philadelphia Eagles', 'opponent': 'vs WAS', 'best_rank': 5, 'worst_rank': 11, 'avg_rank': 7.2},
    {'rank': 8, 'team_abbr': 'PIT', 'team_name': 'Pittsburgh Steelers', 'opponent': 'vs CLE', 'best_rank': 6, 'worst_rank': 12, 'avg_rank': 8.5},
    {'rank': 9, 'team_abbr': 'NYJ', 'team_name': 'New York Jets', 'opponent': 'vs NE', 'best_rank': 7, 'worst_rank': 14, 'avg_rank': 9.3},
    {'rank': 10, 'team_abbr': 'MIA', 'team_name': 'Miami Dolphins', 'opponent': 'vs BUF', 'best_rank': 8, 'worst_rank': 15, 'avg_rank': 10.6},
    {'rank': 11, 'team_abbr': 'DET', 'team_name': 'Detroit Lions', 'opponent': 'vs MIN', 'best_rank': 9, 'worst_rank': 16, 'avg_rank': 11.4},
    {'rank': 12, 'team_abbr': 'NO', 'team_name': 'New Orleans Saints', 'opponent': 'vs ATL', 'best_rank': 10, 'worst_rank': 17, 'avg_rank': 12.7},
    {'rank': 13, 'team_abbr': 'LAC', 'team_name': 'Los Angeles Chargers', 'opponent': 'vs KC', 'best_rank': 11, 'worst_rank': 18, 'avg_rank': 13.9},
    {'rank': 14, 'team_abbr': 'NE', 'team_name': 'New England Patriots', 'opponent': 'vs NYJ', 'best_rank': 12, 'worst_rank': 19, 'avg_rank': 14.5},
    {'rank': 15, 'team_abbr': 'GB', 'team_name': 'Green Bay Packers', 'opponent': 'vs SF', 'best_rank': 13, 'worst_rank': 20, 'avg_rank': 15.8},
    {'rank': 16, 'team_abbr': 'SEA', 'team_name': 'Seattle Seahawks', 'opponent': 'vs LA', 'best_rank': 14, 'worst_rank': 21, 'avg_rank': 16.3},
    {'rank': 17, 'team_abbr': 'DEN', 'team_name': 'Denver Broncos', 'opponent': 'vs BAL', 'best_rank': 15, 'worst_rank': 22, 'avg_rank': 17.6},
    {'rank': 18, 'team_abbr': 'TB', 'team_name': 'Tampa Bay Buccaneers', 'opponent': 'vs CAR', 'best_rank': 16, 'worst_rank': 23, 'avg_rank': 18.4},
    {'rank': 19, 'team_abbr': 'MIN', 'team_name': 'Minnesota Vikings', 'opponent': 'vs DET', 'best_rank': 17, 'worst_rank': 24, 'avg_rank': 19.2},
    {'rank': 20, 'team_abbr': 'TEN', 'team_name': 'Tennessee Titans', 'opponent': 'vs IND', 'best_rank': 18, 'worst_rank': 25, 'avg_rank': 20.5},
    {'rank': 21, 'team_abbr': 'LA', 'team_name': 'Los Angeles Rams', 'opponent': 'vs SEA', 'best_rank': 19, 'worst_rank': 26, 'avg_rank': 21.3},
    {'rank': 22, 'team_abbr': 'HOU', 'team_name': 'Houston Texans', 'opponent': 'vs JAX', 'best_rank': 20, 'worst_rank': 27, 'avg_rank': 22.7},
    {'rank': 23, 'team_abbr': 'CIN', 'team_name': 'Cincinnati Bengals', 'opponent': 'vs CHI', 'best_rank': 21, 'worst_rank': 28, 'avg_rank': 23.1},
    {'rank': 24, 'team_abbr': 'IND', 'team_name': 'Indianapolis Colts', 'opponent': 'vs TEN', 'best_rank': 22, 'worst_rank': 29, 'avg_rank': 24.6},
    {'rank': 25, 'team_abbr': 'ATL', 'team_name': 'Atlanta Falcons', 'opponent': 'vs NO', 'best_rank': 23, 'worst_rank': 30, 'avg_rank': 25.4},
    {'rank': 26, 'team_abbr': 'LV', 'team_name': 'Las Vegas Raiders', 'opponent': 'vs ARI', 'best_rank': 24, 'worst_rank': 31, 'avg_rank': 26.8},
    {'rank': 27, 'team_abbr': 'WAS', 'team_name': 'Washington Commanders', 'opponent': 'vs PHI', 'best_rank': 25, 'worst_rank': 32, 'avg_rank': 27.2},
    {'rank': 28, 'team_abbr': 'CHI', 'team_name': 'Chicago Bears', 'opponent': 'vs CIN', 'best_rank': 26, 'worst_rank': 32, 'avg_rank': 28.5},
    {'rank': 29, 'team_abbr': 'JAX', 'team_name': 'Jacksonville Jaguars', 'opponent': 'vs HOU', 'best_rank': 27, 'worst_rank': 32, 'avg_rank': 29.3},
    {'rank': 30, 'team_abbr': 'NYG', 'team_name': 'New York Giants', 'opponent': 'vs DAL', 'best_rank': 28, 'worst_rank': 32, 'avg_rank': 30.1},
    {'rank': 31, 'team_abbr': 'CAR', 'team_name': 'Carolina Panthers', 'opponent': 'vs TB', 'best_rank': 29, 'worst_rank': 32, 'avg_rank': 31.2},
    {'rank': 32, 'team_abbr': 'ARI', 'team_name': 'Arizona Cardinals', 'opponent': 'vs LV', 'best_rank': 30, 'worst_rank': 32, 'avg_rank': 31.8},
]

# Add timestamp and week info
for ranking in defense_rankings_manual:
    ranking['fetched_at'] = datetime.now()
    ranking['season'] = 2025
    ranking['week'] = 18  # UPDATE THIS WEEKLY

df_rankings = pd.DataFrame(defense_rankings_manual)

print(f"\n✅ Loaded {len(df_rankings)} defense rankings")
print(f"   Season: {df_rankings['season'].iloc[0]}")
print(f"   Week: {df_rankings['week'].iloc[0]}")
print(f"\n📝 NOTE: Update the rankings list + season/week in this cell weekly")
print("="*80)

display(df_rankings.head(10))

In [0]:
# Parse the rankings table
defense_rankings = []

if table:
    rows = table.find('tbody').find_all('tr')
    print(f"Found {len(rows)} defense rankings\n")
    
    for row in rows:
        try:
            cols = row.find_all('td')
            
            # Extract rank
            rank_cell = cols[0]
            rank = int(rank_cell.get_text(strip=True))
            
            # Extract team name
            team_cell = cols[1]
            team_link = team_cell.find('a')
            if team_link:
                team_full_name = team_link.get_text(strip=True)
                # Extract abbreviation from the team name or link
                team_abbr = team_link.get('class', [''])[0].split('-')[-1].upper() if team_link.get('class') else ''
            else:
                team_full_name = team_cell.get_text(strip=True)
                team_abbr = ''
            
            # Extract opponent (if available)
            opponent = cols[2].get_text(strip=True) if len(cols) > 2 else ''
            
            # Extract best/worst rank range
            best_worst = cols[3].get_text(strip=True) if len(cols) > 3 else ''
            if '-' in best_worst:
                best_rank, worst_rank = best_worst.split('-')
                best_rank = int(best_rank.strip())
                worst_rank = int(worst_rank.strip())
            else:
                best_rank = worst_rank = rank
            
            # Extract average rank
            avg_rank = float(cols[4].get_text(strip=True)) if len(cols) > 4 else rank
            
            defense_rankings.append({
                'rank': rank,
                'team_name': team_full_name,
                'team_abbr': team_abbr,
                'opponent': opponent,
                'best_rank': best_rank,
                'worst_rank': worst_rank,
                'avg_rank': avg_rank,
                'fetched_at': datetime.now()
            })
            
        except Exception as e:
            print(f"Error parsing row: {e}")
            continue
    
    # Create DataFrame
    df_rankings = pd.DataFrame(defense_rankings)
    print(f"\n✅ Parsed {len(df_rankings)} defense rankings")
    display(df_rankings.head(15))
else:
    print("❌ No table found to parse")
    df_rankings = pd.DataFrame()

In [0]:
# Map team names to standard abbreviations
team_mapping = {
    'Arizona Cardinals': 'ARI',
    'Atlanta Falcons': 'ATL',
    'Baltimore Ravens': 'BAL',
    'Buffalo Bills': 'BUF',
    'Carolina Panthers': 'CAR',
    'Chicago Bears': 'CHI',
    'Cincinnati Bengals': 'CIN',
    'Cleveland Browns': 'CLE',
    'Dallas Cowboys': 'DAL',
    'Denver Broncos': 'DEN',
    'Detroit Lions': 'DET',
    'Green Bay Packers': 'GB',
    'Houston Texans': 'HOU',
    'Indianapolis Colts': 'IND',
    'Jacksonville Jaguars': 'JAX',
    'Kansas City Chiefs': 'KC',
    'Las Vegas Raiders': 'LV',
    'Los Angeles Chargers': 'LAC',
    'Los Angeles Rams': 'LA',
    'Miami Dolphins': 'MIA',
    'Minnesota Vikings': 'MIN',
    'New England Patriots': 'NE',
    'New Orleans Saints': 'NO',
    'New York Giants': 'NYG',
    'New York Jets': 'NYJ',
    'Philadelphia Eagles': 'PHI',
    'Pittsburgh Steelers': 'PIT',
    'San Francisco 49ers': 'SF',
    'Seattle Seahawks': 'SEA',
    'Tampa Bay Buccaneers': 'TB',
    'Tennessee Titans': 'TEN',
    'Washington Commanders': 'WAS'
}

# Normalize team abbreviations
if not df_rankings.empty:
    # Try to map full names to abbreviations
    df_rankings['team_abbr_clean'] = df_rankings['team_name'].map(team_mapping)
    
    # If mapping failed, try to extract from existing team_abbr
    df_rankings['team_abbr_clean'] = df_rankings['team_abbr_clean'].fillna(df_rankings['team_abbr'])
    
    # Show any unmapped teams
    unmapped = df_rankings[df_rankings['team_abbr_clean'].isna()]
    if not unmapped.empty:
        print("\n⚠️ Unmapped teams:")
        print(unmapped[['rank', 'team_name', 'team_abbr']])
    
    # Use the cleaned abbreviation
    df_rankings['team_abbr'] = df_rankings['team_abbr_clean']
    df_rankings.drop('team_abbr_clean', axis=1, inplace=True)
    
    print(f"\n✅ Normalized {df_rankings['team_abbr'].notna().sum()} team abbreviations")
    display(df_rankings[['rank', 'team_abbr', 'team_name', 'opponent', 'avg_rank']].head(10))

In [0]:
# Create tables if they don't exist, then save rankings
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

print("\n" + "="*80)
print("Saving Defense Rankings to Delta Tables")
print("="*80)

if not df_rankings.empty:
    # Convert to Spark DataFrame
    spark_df = spark.createDataFrame(df_rankings)
    
    # Create or replace current rankings table
    print("\n📝 Saving to main.fantasai.defense_rankings_current (overwrite)...")
    spark_df.write.mode('overwrite').option('overwriteSchema', 'true').saveAsTable('main.fantasai.defense_rankings_current')
    print(f"✅ Saved {len(df_rankings)} rankings")
    
    # Append to historical log
    print("\n📝 Appending to main.fantasai.defense_rankings_history...")
    try:
        spark_df.write.mode('append').saveAsTable('main.fantasai.defense_rankings_history')
        print("✅ Appended to history")
    except Exception as e:
        # If history table doesn't exist, create it
        if 'TABLE_OR_VIEW_NOT_FOUND' in str(e) or 'does not exist' in str(e):
            print("   History table doesn't exist, creating it...")
            spark_df.write.mode('overwrite').saveAsTable('main.fantasai.defense_rankings_history')
            print("✅ Created history table with initial data")
        else:
            print(f"⚠️  Error appending to history: {e}")
    
    print("\n" + "="*80)
    print("✅ DEFENSE RANKINGS SAVED SUCCESSFULLY")
    print("="*80)
    print(f"\nTables:")
    print(f"  • main.fantasai.defense_rankings_current ({len(df_rankings)} rows)")
    print(f"  • main.fantasai.defense_rankings_history (appended)")
else:
    print("\n❌ No rankings data to save")

In [0]:
%sql
-- Preview the saved rankings
SELECT 
  rank,
  team_abbr,
  team_name,
  opponent,
  best_rank,
  worst_rank,
  ROUND(avg_rank, 1) as avg_rank,
  season,
  week,
  fetched_at
FROM main.fantasai.defense_rankings_current
ORDER BY rank
LIMIT 15